# Vocational NMT: IndicTrans2 domain adaptation (electrician)

1. **Drafts:** IndicTrans2-1B (teacher) translates the 24.6k English textbook sentences into hi / gom / mai / doi. This step is skipped when the drafts already exist (see below).
2. **Fine-tune:** a joint LoRA adapter on IndicTrans2 dist-200M (the student). Meanwhile, GPU 1 scores the base model on IN22-Gen.
3. **Domain eval:** base vs LoRA vs teacher on the test sets.
4. **IN22-Gen forgetting check** for the LoRA model.

**Before running:**
* **Accelerator:** GPU T4 x2.
* **Internet:** on.
* **Secret:** `HF_TOKEN`. Accept the terms on Hugging Face for both models **and** the dataset `ai4bharat/IN22-Gen`. If the token is fine-grained, enable *"Read access to contents of all public gated repos you can access"*.
* **Input:** the bundle dataset (`vocational-nmt-bundle.zip`).
* **Optional:** to reuse drafts from an earlier run, add that run's output (or a dataset with its `drafts_*.csv`) as an input.

Use **Save Version → Save & Run All**. Every step stops the run on error, and `/kaggle/working/outputs.zip` is written at the end.

In [ ]:
# 1. Copy the bundle into a writable working directory (+ reuse earlier drafts if provided)
import os, shutil, subprocess, sys, glob, zipfile
from pathlib import Path

PROJECT = Path("/kaggle/working/project")
if not PROJECT.exists():
    # Only folders/zips carrying kaggle/BUNDLE_VERSION count as code; outputs of earlier runs
    # (added as inputs to reuse drafts) also contain a backend/ folder with OLD code.
    candidates = []
    for z in glob.glob("/kaggle/input/**/vocational-nmt-bundle.zip", recursive=True):
        with zipfile.ZipFile(z) as zf:
            if "kaggle/BUNDLE_VERSION" in zf.namelist():
                candidates.append((zf.read("kaggle/BUNDLE_VERSION").decode().strip(), z))
    for stamp in glob.glob("/kaggle/input/**/kaggle/BUNDLE_VERSION", recursive=True):
        candidates.append((Path(stamp).read_text().strip(), str(Path(stamp).parents[1])))
    if not candidates:
        raise FileNotFoundError("No bundle with kaggle/BUNDLE_VERSION under /kaggle/input - upload the latest bundle zip.")
    version, source = max(candidates)
    print(f"Using bundle {version} from {source}")
    if source.endswith(".zip"):
        zipfile.ZipFile(source).extractall(PROJECT)
    else:
        shutil.copytree(source, PROJECT)
print("bundle version:", (PROJECT / "kaggle/BUNDLE_VERSION").read_text().strip())
os.chdir(PROJECT)
os.environ["PYTHONPATH"] = str(PROJECT)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation during beam search
sys.path.insert(0, str(PROJECT))

CORPUS = PROJECT / "backend/data/corpus_v2"
for pattern, sub in (("drafts_*.csv", ""), ("postedit/test_*.csv", "postedit")):
    for src in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        dst = CORPUS / sub / Path(src).name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(src, dst)
            print("reused", src)
print(sorted(p.name for p in CORPUS.iterdir()))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install the pinned stack (about 1 minute)
!pip install -q -r kaggle/requirements-kaggle.txt
!python -c "import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)"

In [ ]:
# 3. Hugging Face token + access check for the gated repos
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import hf_hub_download
IN22_OK = True
for repo, kind, fname in [("ai4bharat/indictrans2-en-indic-dist-200M", "model", "config.json"),
                          ("ai4bharat/indictrans2-en-indic-1B", "model", "config.json"),
                          ("ai4bharat/IN22-Gen", "dataset", "data/train-00000-of-00001.parquet")]:
    try:
        hf_hub_download(repo, fname, repo_type=kind, token=os.environ["HF_TOKEN"])
        print("access OK:", repo)
    except Exception as exc:
        print(f"NO ACCESS: {repo} -> accept its terms on huggingface.co ({type(exc).__name__})")
        if kind == "model":
            raise
        IN22_OK = False
print("IN22-Gen available:", IN22_OK)

In [ ]:
# 4. Inference smoke test
def check(step):
    assert _exit_code == 0, f"{step} failed - see the output above"

!python -m backend.baseline_inference --text "Check the insulation resistance with a megger before energising the circuit." --target-lang hin_Deva
check("base inference")

## Step 1: teacher drafts
About 1 hour on 2×T4 (13–14 sent/s per GPU). Languages whose `drafts_*.csv` already exist are skipped.

In [ ]:
import torch
n_gpu = torch.cuda.device_count()
groups = [["hin_Deva", "mai_Deva"], ["gom_Deva", "doi_Deva"]] if n_gpu >= 2 else [["hin_Deva", "mai_Deva", "gom_Deva", "doi_Deva"]]
procs = []
for gpu, langs in enumerate(groups):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = open(f"/kaggle/working/drafts_gpu{gpu}.log", "w")
    procs.append(subprocess.Popen([sys.executable, "-m", "backend.generate_drafts", "--langs", ",".join(langs)],
                                  env=env, stdout=log, stderr=subprocess.STDOUT))
codes = [p.wait() for p in procs]
for gpu in range(len(groups)):
    print(open(f"/kaggle/working/drafts_gpu{gpu}.log").read()[-800:])
assert all(c == 0 for c in codes), codes
!ls -la backend/data/corpus_v2 backend/data/corpus_v2/postedit

In [ ]:
# Training smoke test (~3 min): 10 steps + a dev evaluation, then load the resulting adapter
!CUDA_VISIBLE_DEVICES=0 python -m backend.finetune --run-name smoke --max-steps 10 --eval-steps 5 --dev-per-lang 8
check("training smoke test")
!python -m backend.baseline_inference --model backend/models/it2_lora_smoke --text "Connect the neutral wire to the terminal block." --target-lang mai_Deva
check("adapter loading")
shutil.rmtree("backend/models/it2_lora_smoke", ignore_errors=True)

## Step 2: LoRA fine-tuning on GPU 0 (≤ 4,515 steps, early stopping), with base IN22-Gen on GPU 1

In [ ]:
in22_base = None
if IN22_OK and n_gpu >= 2:
    in22_base = subprocess.Popen(
        [sys.executable, "-m", "backend.run_baseline_eval", "--output", "backend/results/in22_gen_scores.csv"],
        env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"),
        stdout=open("/kaggle/working/in22_base.log", "w"), stderr=subprocess.STDOUT)

!CUDA_VISIBLE_DEVICES=0 python -m backend.finetune --run-name joint
check("fine-tuning")

## Step 3: domain evaluation
Until the post-edited sheets exist, references are teacher drafts and the files are tagged `UNVERIFIED`. Use them for sanity checks only, not for the paper.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m backend.run_domain_eval --adapter backend/models/it2_lora_joint
check("domain evaluation")

## Step 4: IN22-Gen (Table 1 + forgetting check)

In [ ]:
if not IN22_OK:
    print("Skipped: no access to ai4bharat/IN22-Gen.")
else:
    if in22_base is not None:
        print("base IN22 exit code:", in22_base.wait())
        print(open("/kaggle/working/in22_base.log").read()[-2500:])
    else:
        !CUDA_VISIBLE_DEVICES=0 python -m backend.run_baseline_eval --output backend/results/in22_gen_scores.csv
    !CUDA_VISIBLE_DEVICES=0 python -m backend.run_baseline_eval --skip-base --adapter backend/models/it2_lora_joint --output backend/results/in22_gen_lora.csv

In [ ]:
# Package everything that must come back to your machine
out = Path("/kaggle/working/outputs")
shutil.rmtree(out, ignore_errors=True)
for src, dst in [("backend/data/corpus_v2", "corpus_v2"), ("backend/models/it2_lora_joint", "it2_lora_joint"),
                 ("backend/results", "results")]:
    if Path(src).exists():
        shutil.copytree(src, out / dst, ignore=shutil.ignore_patterns("checkpoints"))
    else:
        print("missing:", src)
for log in glob.glob("/kaggle/working/*.log"):
    shutil.copy(log, out)
shutil.make_archive("/kaggle/working/outputs", "zip", out)
!du -sh /kaggle/working/outputs.zip